<div align="center">

# AI-Powered Real-Time Noise Cancellation Adaptive System for Multiple Speaker

### Stage A: Google Colab audio classification pipeline

**Three classes | 16 kHz mono audio | 2 second windows | CNN on mel-spectrograms**

</div>

This notebook prepares the classifier that will later be integrated into the MATLAB real-time system. It is deliberately limited to **Stage A**. Adaptive noise cancellation, source separation, microphone streaming, and real-time output belong to Stage B and are not implemented here.

| Class ID | Meaning | Audio presented to the classifier |
| --- | --- | --- |
| `0` | Clean Speech | A standardized speech window |
| `1` | Environmental Noise | A standardized environmental-noise window |
| `2` | Noisy Speech | Speech mixed with controlled environmental noise |

**Run order:** execute the cells from top to bottom in a fresh Colab runtime. Begin with `QUICK_TEST_MODE = True` so the complete pipeline can be checked before committing to a long training run. Quick-test metrics are pipeline checks, not final FYP results.

## Technical implementation plan and risk audit

The pipeline splits speakers and complete noise recordings before segmentation, measures SNR after mixing, uses only verified RIR files, fits normalization on training data, and stores configuration/provenance with exported artifacts. These checks target leakage, clipping, invalid archives, normalization leakage, and Colab storage instability.

# Stage 1 - Environment setup

## Step 1 - Install and verify dependencies

Install the audio, data, plotting, machine-learning, and export packages before importing them. The import and version checks make runtime compatibility visible before any dataset work begins.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'librosa', 'soundfile', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'requests', 'tqdm'], check=False)

## Step 2 - Import libraries and check versions

The next two code cells import the pipeline dependencies and report the key runtime versions before dataset work begins.

In [ ]:
import os
import gc
import csv
import json
import math
import random
import hashlib
import shutil
import tarfile
import zipfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.signal import fftconvolve
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
import tensorflow as tf
from IPython.display import Audio, display

## Step 3 - Configure runtime and audio settings

This check records Python, TensorFlow, and librosa versions before audio processing and model training. The following configuration cell then selects the runtime paths and defines the audio experiment settings.

In [ ]:
print('Python:', subprocess.check_output(['python', '--version'], text=True).strip())
print('TensorFlow:', tf.__version__)
print('librosa:', librosa.__version__)
print('PASS: Core imports succeeded.')

# Stage 1 - Environment setup

## Step 4 - Create storage and configuration

The next cell creates persistent and temporary directories, fixes random seeds, and generates a configuration ID for the reproducible audio experiment.

In [ ]:
PROJECT_TITLE = 'AI-Powered Real-Time Noise Cancellation Adaptive System for Multiple Speaker'
QUICK_TEST_MODE = True
IS_COLAB = 'google.colab' in sys.modules
BASE_ROOT = Path('/content') if IS_COLAB else Path.cwd()
DATA_ROOT = BASE_ROOT / 'fyp_noise_data'
RAW_ROOT = DATA_ROOT / 'raw'
EXTERNAL_DATASET_ROOT = DATA_ROOT / 'external'
ARTIFACT_ROOT = BASE_ROOT / 'artifacts'
CONFIG_DIR = ARTIFACT_ROOT / 'configuration'
METADATA_DIR = ARTIFACT_ROOT / 'results' / 'metadata'
MANIFEST_DIR = ARTIFACT_ROOT / 'results' / 'manifests'
RESULT_DIR = ARTIFACT_ROOT / 'results'
PLOT_DIR = RESULT_DIR / 'plots'
MODEL_DIR = ARTIFACT_ROOT / 'models'
CHECKPOINT_DIR = ARTIFACT_ROOT / 'checkpoints'
MATLAB_DIR = ARTIFACT_ROOT / 'matlab'
EXPORT_DIR = ARTIFACT_ROOT / 'exports'
VERIFIED_RIR_DIR = ARTIFACT_ROOT / 'verified_rirs'
SAMPLE_RATE = 16000
CHANNELS = 1
SEGMENT_SECONDS = 2.0
SAMPLES_PER_SEGMENT = int(SAMPLE_RATE * SEGMENT_SECONDS)
SNR_LEVELS_DB = [-5.0, 0.0, 5.0, 10.0, 15.0]
FEATURE_CONFIG = {'n_fft': 512, 'hop_length': 256, 'win_length': 512, 'n_mels': 64, 'fmin': 20, 'fmax': 8000, 'power': 2.0, 'center': True, 'db_conversion': 'power_to_db'}
MAX_SPEECH_FILES = 120 if QUICK_TEST_MODE else 2000
MAX_NOISE_FILES = 120 if QUICK_TEST_MODE else 1500
SEGMENTS_PER_SPEECH_FILE = 2
SEGMENTS_PER_NOISE_FILE = 2
MAX_RECORDS_PER_CLASS_SPLIT = 300 if QUICK_TEST_MODE else 5000
RIR_PROBABILITY = 0.25
BATCH_SIZE = 32
EPOCHS = 2 if QUICK_TEST_MODE else 30
CLASS_NAMES = {0: 'Clean Speech', 1: 'Environmental Noise', 2: 'Noisy Speech'}
DATASET_PLAN_VERSION = 'automatic-public-sources-v2-mini-or-librispeech100-with-musan-demand-esc50-slr28'
for folder in [RAW_ROOT, EXTERNAL_DATASET_ROOT, CONFIG_DIR, METADATA_DIR, MANIFEST_DIR, RESULT_DIR, PLOT_DIR, MODEL_DIR, CHECKPOINT_DIR, MATLAB_DIR, EXPORT_DIR, VERIFIED_RIR_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
SEED = 2025
random.seed(SEED)
np.random.seed(SEED)
CONFIG_ID = hashlib.sha256(f'{DATASET_PLAN_VERSION}|{QUICK_TEST_MODE}|{SEED}'.encode()).hexdigest()[:12]
config = {'project_title': PROJECT_TITLE, 'quick_test_mode': QUICK_TEST_MODE, 'is_colab': IS_COLAB, 'seed': SEED, 'config_id': CONFIG_ID, 'feature_config': FEATURE_CONFIG}
(CONFIG_DIR / f'config_{CONFIG_ID}.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
print('Runtime:', 'Colab' if IS_COLAB else 'Local', '| Mode:', 'QUICK' if QUICK_TEST_MODE else 'FULL')
print('Configuration:', CONFIG_ID, '| Storage root:', DATA_ROOT)
print('PASS: Environment configuration saved.')

## Step 4.1 - Runtime paths and mode

This cell separates environment detection from experiment settings. `QUICK_TEST_MODE` selects the small Mini LibriSpeech plus ESC-50 path; full mode selects train-clean-100 and available automatic noise sources.

# Stage 2 - Dataset acquisition

## Step 5 - Explain the dataset portfolio

Mini LibriSpeech and ESC-50 are acquired automatically for the quick test. Full mode automatically acquires LibriSpeech train-clean-100, MUSAN, DEMAND through the public Zenodo API record, ESC-50, and SLR28. Full mode uses train-clean-100 as clean speech and combines available MUSAN, DEMAND, and ESC-50 noise. DNS, WHAM, WHAMR, and LibriMix remain documented resources with their roles and access limits; LibriMix is reserved for future multi-speaker evaluation and MATLAB integration. License documentation is separate from download capability.

## Step 6 - Define and acquire datasets

The registry records role, format, license, access, and acquisition capability for every source. Public sources are downloaded automatically in the selected mode; existing verified files are reused. License documentation is maintained separately from download capability, and failures report their actual reason.

In [ ]:
DATASET_SOURCES = {
    'Mini LibriSpeech': {'role': 'clean_speech', 'kind': 'speech', 'quick_use': True, 'full_use': False, 'acquisition': 'AUTOMATIC', 'direct_url': 'https://www.openslr.org/resources/31/train-clean-5.tar.gz', 'archive_name': 'train-clean-5.tar.gz', 'license_note': 'LibriSpeech terms require verification; see LICENSES_AND_DATASETS.md.', 'access_note': 'Public OpenSLR download; license documentation is separate from download capability.', 'expected_formats': ['flac'], 'local_path': RAW_ROOT / 'mini_librispeech'},
    'LibriSpeech train-clean-100': {'role': 'clean_speech', 'kind': 'speech', 'quick_use': False, 'full_use': True, 'acquisition': 'AUTOMATIC', 'direct_url': 'https://www.openslr.org/resources/12/train-clean-100.tar.gz', 'archive_name': 'train-clean-100.tar.gz', 'license_note': 'LibriSpeech terms require verification; see LICENSES_AND_DATASETS.md.', 'access_note': 'Public OpenSLR download; license documentation is separate from download capability.', 'expected_formats': ['flac'], 'local_path': EXTERNAL_DATASET_ROOT / 'train-clean-100'},
    'MUSAN': {'role': 'noise', 'kind': 'noise', 'quick_use': False, 'full_use': True, 'acquisition': 'AUTOMATIC', 'direct_url': 'https://openslr.trmal.net/resources/17/musan.tar.gz', 'archive_name': 'musan.tar.gz', 'license_note': 'MUSAN license terms require verification; see LICENSES_AND_DATASETS.md.', 'access_note': 'Public OpenSLR mirror download; license documentation is separate from download capability.', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'musan'},
    'DEMAND': {'role': 'noise', 'kind': 'noise', 'quick_use': False, 'full_use': True, 'acquisition': 'AUTOMATIC', 'direct_url': 'https://zenodo.org/api/records/1227121', 'archive_name': '', 'license_note': 'DEMAND license terms require verification; see LICENSES_AND_DATASETS.md.', 'access_note': 'Public Zenodo record API; the downloadable file is resolved from the API files list.', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'demand'},
    'ESC-50': {'role': 'noise', 'kind': 'noise', 'quick_use': True, 'full_use': True, 'acquisition': 'AUTOMATIC', 'direct_url': 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip', 'archive_name': 'esc50.zip', 'license_note': 'CC BY-NC; verify intended use; see LICENSES_AND_DATASETS.md.', 'access_note': 'Public GitHub download; license documentation is separate from download capability.', 'expected_formats': ['wav', 'csv'], 'local_path': RAW_ROOT / 'esc50'},
    'DNS': {'role': 'noisy_speech', 'kind': 'noisy_speech', 'quick_use': False, 'full_use': False, 'acquisition': 'MANUAL_REQUIRED', 'direct_url': 'https://dnschallenge.microsoft.com/', 'archive_name': '', 'license_note': 'DNS access terms require verification.', 'access_note': 'No stable unrestricted public programmatic download is provided; manual access may be required.', 'expected_formats': ['wav', 'json'], 'local_path': EXTERNAL_DATASET_ROOT / 'dns'},
    'WHAM': {'role': 'noisy_speech', 'kind': 'noisy_speech', 'quick_use': False, 'full_use': False, 'acquisition': 'MANUAL_REQUIRED', 'direct_url': 'https://wham.whisper.ai/', 'archive_name': '', 'license_note': 'WHAM terms require verification.', 'access_note': 'No stable unrestricted public direct download is provided; manual acquisition may be required.', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'wham'},
    'WHAMR': {'role': 'reverb_noisy', 'kind': 'noisy_speech', 'quick_use': False, 'full_use': False, 'acquisition': 'MANUAL_REQUIRED', 'direct_url': 'https://wham.whisper.ai/', 'archive_name': '', 'license_note': 'WHAMR terms require verification.', 'access_note': 'No stable unrestricted public direct download is provided; manual acquisition may be required.', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'whamr'},
    'LibriMix': {'role': 'future_multi_speaker_evaluation', 'kind': 'multi_speaker', 'quick_use': False, 'full_use': False, 'acquisition': 'MANUAL_REQUIRED', 'direct_url': 'https://github.com/JorisCos/LibriMix', 'archive_name': '', 'license_note': 'LibriMix terms require verification.', 'access_note': 'Future multi-speaker evaluation resource, especially for MATLAB experiments; no stable direct download is assumed.', 'expected_formats': ['wav', 'json'], 'local_path': EXTERNAL_DATASET_ROOT / 'librimix'},
    'SLR28': {'role': 'rir', 'kind': 'rir', 'quick_use': False, 'full_use': True, 'acquisition': 'AUTOMATIC', 'direct_url': 'https://www.openslr.org/resources/28/rirs_noises.zip', 'archive_name': 'rirs_noises.zip', 'license_note': 'SLR28 terms require verification; see LICENSES_AND_DATASETS.md.', 'access_note': 'Public OpenSLR download; only verified RIR files are copied to VERIFIED_RIR_DIR.', 'expected_formats': ['wav'], 'local_path': EXTERNAL_DATASET_ROOT / 'slr28'}
}
DATASET_REGISTRY = list(DATASET_SOURCES)

### Dataset registry

The registry is definition-only. `AUTOMATIC` marks the public six sources, while DNS, WHAM, WHAMR, and LibriMix remain honest `MANUAL_REQUIRED` resources.

In [ ]:
def download_file(url, destination, retries=3):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 1024:
        return destination
    last_error = None
    for attempt in range(1, retries + 1):
        temporary = destination.with_name(destination.name + '.part')
        try:
            with requests.get(url, stream=True, timeout=(20, 120), headers={'User-Agent': 'FYP-Colab-Pipeline/1.0'}) as response:
                response.raise_for_status()
                with temporary.open('wb') as output:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            output.write(chunk)
            if temporary.stat().st_size <= 1024:
                raise RuntimeError('Downloaded file is unexpectedly small.')
            temporary.replace(destination)
            return destination
        except Exception as error:
            last_error = error
            if temporary.exists():
                temporary.unlink()
    raise RuntimeError(f'Unable to download {url} after {retries} attempts: {last_error}')

### Download helper

This helper streams public archives into a temporary file and replaces the destination only after a non-empty download succeeds.

### Archive verification

Verification checks archive readability and CRC information before extraction. A `PASS` means the container is readable, not that its contents are complete.

In [ ]:
def verify_archive(archive):
    archive = Path(archive)
    if not archive.exists() or archive.stat().st_size <= 1024:
        raise RuntimeError(f'Archive is missing or unexpectedly small: {archive}')
    try:
        if archive.name.endswith(('.tar.gz', '.tgz')):
            with tarfile.open(archive, 'r:*') as handle:
                if not handle.getmembers():
                    raise RuntimeError('tar archive contains no members')
        elif archive.suffix.lower() == '.zip':
            with zipfile.ZipFile(archive) as handle:
                if handle.testzip() is not None or not handle.namelist():
                    raise RuntimeError('zip archive failed CRC or contains no members')
        else:
            raise ValueError(f'Unsupported archive type: {archive}')
    except (tarfile.TarError, zipfile.BadZipFile) as error:
        raise RuntimeError(f'Archive verification failed: {error}') from error
    return archive

### Safe extraction

The extraction guard rejects absolute and parent-traversal paths before writing files. The marker makes reruns idempotent.

In [ ]:
def extract_archive(archive, extract_dir):
    extract_dir = Path(extract_dir)
    marker = extract_dir / '.extracted_ok'
    if marker.exists():
        return extract_dir
    extract_dir.mkdir(parents=True, exist_ok=True)
    verify_archive(archive)
    target = extract_dir.resolve()
    if str(archive).endswith(('.tar.gz', '.tgz')):
        with tarfile.open(archive, 'r:*') as handle:
            for member in handle.getmembers():
                member_path = (target / member.name).resolve()
                if target not in member_path.parents and member_path != target:
                    raise RuntimeError(f'Unsafe tar member path: {member.name}')
            handle.extractall(target)
    else:
        with zipfile.ZipFile(archive) as handle:
            for member in handle.infolist():
                member_path = (target / member.filename).resolve()
                if target not in member_path.parents and member_path != target:
                    raise RuntimeError(f'Unsafe zip member path: {member.filename}')
            handle.extractall(target)
    marker.write_text('ok', encoding='utf-8')
    return extract_dir

### File counts and DEMAND resolution

Counts provide inventory evidence. DEMAND is resolved from the public Zenodo record rather than assuming a fixed filename.

In [ ]:
def audio_file_count(root, formats):
    root = Path(root)
    return sum(1 for extension in formats for path in root.rglob(f'*.{extension}') if path.is_file()) if root.exists() else 0

def expected_files_exist(root, formats):
    return audio_file_count(root, formats) > 0

def resolve_demand_file(record_url):
    response = requests.get(record_url, timeout=(20, 60), headers={'User-Agent': 'FYP-Colab-Pipeline/1.0'})
    response.raise_for_status()
    candidates = []
    for item in response.json().get('files', []):
        key = str(item.get('key', item.get('filename', '')))
        url = item.get('links', {}).get('self') or item.get('links', {}).get('content')
        if url and key.lower().endswith(('.zip', '.tar.gz', '.tgz', '.wav')):
            normalized = key.lower().replace('-', '').replace('_', '')
            preference = 0 if any(token in normalized for token in ('16khz', '16k', '16000')) else 1
            candidates.append((preference, key, url))
    if not candidates:
        raise RuntimeError('Zenodo DEMAND record contains no downloadable archive/audio file in its files list.')
    _, filename, url = sorted(candidates)[0]
    return url, filename

### RIR validation and collection

Only readable WAV files with valid audio metadata are copied into the verified RIR directory. An empty result means reverberation augmentation is unavailable.

In [ ]:
def verify_rir_file(path):
    try:
        info = sf.info(str(path))
        return path.suffix.lower() == '.wav' and info.frames > 0 and info.channels >= 1 and info.samplerate > 0
    except Exception:
        return False

def verify_and_collect_rirs(source_root):
    verified = []
    VERIFIED_RIR_DIR.mkdir(parents=True, exist_ok=True)
    for path in Path(source_root).rglob('*.wav'):
        if verify_rir_file(path):
            destination = VERIFIED_RIR_DIR / path.name
            if not destination.exists():
                shutil.copy2(path, destination)
            verified.append(destination)
    return sorted(set(verified))

### Acquisition policy and run

Automatic sources are downloaded only when selected by mode; cached verified files are reused. DNS, WHAM, WHAMR, and LibriMix are reported as manual or future resources.

In [ ]:
def acquire_datasets():
    acquired = {}
    for name, source in DATASET_SOURCES.items():
        source['status'] = 'PENDING'
        source['failure_reason'] = ''
        if source['acquisition'] != 'AUTOMATIC':
            source['status'] = 'MANUAL_REQUIRED'
            continue
        if QUICK_TEST_MODE and not source['quick_use']:
            source['status'] = 'SKIPPED_IN_QUICK_TEST'
            continue
        try:
            archive_name = source['archive_name']
            url = source['direct_url']
            if name == 'DEMAND':
                url, archive_name = resolve_demand_file(url)
                source['resolved_url'] = url
                source['archive_name'] = archive_name
            archive = RAW_ROOT / archive_name
            if not expected_files_exist(source['local_path'], source['expected_formats']):
                download_file(url, archive)
                extract_archive(archive, source['local_path'])
            if name == 'SLR28':
                verified = verify_and_collect_rirs(source['local_path'])
                if not verified:
                    raise RuntimeError('No verified WAV RIR files were found in SLR28.')
                source['verified_rir_count'] = len(verified)
            elif not expected_files_exist(source['local_path'], source['expected_formats']):
                raise RuntimeError(f'No expected files found under {source["local_path"]}.')
            source['status'] = 'AVAILABLE'
            acquired[name] = source
        except Exception as error:
            source['status'] = 'FAILED'
            source['failure_reason'] = str(error)
            print(f'FAILED TO DOWNLOAD: {name}: {error}')
    return acquired

ACQUIRED = acquire_datasets()

### Inventory construction and save

The inventory records status, file counts, intended role, paths, URLs, and access notes. It is the audit table for acquisition.

In [ ]:
def inventory_status(source):
    return source.get('status', 'MANUAL_REQUIRED')

dataset_inventory = pd.DataFrame([
    {
        'Dataset': name,
        'Download capability': source['acquisition'],
        'Acquisition status': inventory_status(source),
        'Number of files': audio_file_count(source['local_path'], source['expected_formats']),
        'Role': source['role'],
        'Used for training?': False,
        'Used for validation?': False,
        'Used for testing?': False,
        'Reason if not used': source.get('failure_reason') or ('Quick mode excludes full-mode source.' if source.get('status') == 'SKIPPED_IN_QUICK_TEST' else ''),
        'local path': str(source['local_path']),
        'source URL': source.get('resolved_url', source['direct_url']),
        'license/access': f"{source['license_note']} {source['access_note']}"
    }
    for name, source in DATASET_SOURCES.items()
])
dataset_inventory.to_json(METADATA_DIR / f'dataset_inventory_{CONFIG_ID}.json', orient='records', indent=2)
dataset_inventory.to_csv(METADATA_DIR / f'dataset_inventory_{CONFIG_ID}.csv', index=False)
display(dataset_inventory)

In [ ]:
rir_manifest = pd.DataFrame([
    {'dataset': 'SLR28', 'filepath': str(path), 'verified': True, 'sample_rate': int(sf.info(str(path)).samplerate)}
    for path in sorted(VERIFIED_RIR_DIR.rglob('*.wav')) if verify_rir_file(path)
])
rir_manifest.to_csv(METADATA_DIR / f'rir_manifest_{CONFIG_ID}.csv', index=False)
print('Verified SLR28 RIR files:', len(rir_manifest))

# Stage 3 - Inspect audio and build source manifests

## Step 7 - Inspect and build source manifests

The manifest records paths, measured audio metadata, speaker identity, and noise labels. These source-level records are the basis for auditable splitting and later provenance.

In [ ]:
def audio_metadata(path):
    info = sf.info(str(path))
    return {'sample_rate': int(info.samplerate), 'channels': int(info.channels), 'duration': float(info.duration)}

def infer_speaker_id(path):
    path = Path(path)
    parent_parts = path.parts[:-1]
    known_subsets = {'train-clean-5', 'train-clean-100', 'dev-clean', 'test-clean'}
    subset_index = next((index for index, part in enumerate(parent_parts) if part.lower() in known_subsets), None)
    if subset_index is not None and subset_index + 1 < len(parent_parts):
        return str(parent_parts[subset_index + 1])
    numeric_component = next((part for part in parent_parts if part.isdigit()), None)
    return str(numeric_component) if numeric_component is not None else 'unknown'

### Speech manifest function

Speech records retain speaker IDs and measured audio metadata. Speaker identity is the grouping key used later for leakage-safe splitting.

In [ ]:
def build_speech_manifest(root, limit, dataset_name):
    files = sorted(Path(root).rglob('*.flac')) + sorted(Path(root).rglob('*.wav'))
    grouped_files = {}
    for path in files:
        grouped_files.setdefault(infer_speaker_id(path), []).append(path)
    speaker_ids = sorted(grouped_files)
    if len(speaker_ids) < 3:
        raise ValueError(f'{dataset_name} contains only {len(speaker_ids)} speakers; at least three are required.')
    rng = np.random.default_rng(SEED)
    speaker_order = list(rng.permutation(speaker_ids))
    for speaker_id in speaker_order:
        paths = grouped_files[speaker_id]
        grouped_files[speaker_id] = [paths[index] for index in rng.permutation(len(paths))]
    selected_files = []
    while len(selected_files) < min(limit, len(files)):
        added_file = False
        for speaker_id in speaker_order:
            if grouped_files[speaker_id] and len(selected_files) < limit:
                selected_files.append((speaker_id, grouped_files[speaker_id].pop()))
                added_file = True
        if not added_file:
            break
    return pd.DataFrame([{'dataset': dataset_name, 'speaker_id': str(speaker_id), 'filename': path.name, 'filepath': str(path.resolve()), **audio_metadata(path)} for speaker_id, path in selected_files])

### Noise manifest function

Noise records retain dataset, category, environment, and source identity. Complete noise recordings, rather than derived segments, will be split later.

In [ ]:
def find_esc_metadata(root):
    candidates = list(Path(root).rglob('esc50.csv'))
    if not candidates:
        return {}
    table = pd.read_csv(candidates[0])
    return {str(row.filename): {'noise_category': str(row.category), 'environment': str(row.category)} for _, row in table.iterrows()}

def build_noise_manifest(root, limit, dataset_name):
    root = Path(root)
    esc_metadata = find_esc_metadata(root) if dataset_name == 'ESC-50' else {}
    files = sorted(path for path in root.rglob('*.wav') if path.is_file())
    rows = []
    for path in files[:limit]:
        relative_parts = path.relative_to(root).parts
        labels = esc_metadata.get(path.name, {})
        category = labels.get('noise_category') or (relative_parts[-2] if len(relative_parts) > 1 else 'unknown')
        environment = labels.get('environment') or (relative_parts[-3] if len(relative_parts) > 2 else category)
        rows.append({'dataset': dataset_name, 'noise_source_id': f'{dataset_name}:{path.relative_to(root).as_posix()}', 'filename': path.name, 'filepath': str(path.resolve()), 'noise_category': str(category), 'environment': str(environment), **audio_metadata(path)})
    return pd.DataFrame(rows)

### Active source selection and manifest build

Quick mode uses Mini LibriSpeech and ESC-50. Full mode requires train-clean-100 and uses every available automatic noise source among MUSAN, DEMAND, and ESC-50.

In [ ]:
if QUICK_TEST_MODE:
    speech_dataset_name = 'Mini LibriSpeech'
    noise_dataset_names = ['ESC-50']
else:
    speech_dataset_name = 'LibriSpeech train-clean-100'
    noise_dataset_names = [name for name in ['MUSAN', 'DEMAND', 'ESC-50'] if DATASET_SOURCES[name].get('status') == 'AVAILABLE']
    if DATASET_SOURCES[speech_dataset_name].get('status') != 'AVAILABLE':
        raise RuntimeError('Full mode requires LibriSpeech train-clean-100; automatic acquisition failed.')
    if not noise_dataset_names:
        raise RuntimeError('Full mode requires at least one available automatic noise source.')
speech_source = DATASET_SOURCES[speech_dataset_name]
print('Speech source:', speech_dataset_name)
print('Noise sources:', ', '.join(noise_dataset_names))

In [ ]:
speech_manifest = build_speech_manifest(speech_source['local_path'], MAX_SPEECH_FILES, speech_dataset_name)
noise_tables = [build_noise_manifest(DATASET_SOURCES[name]['local_path'], MAX_NOISE_FILES, name) for name in noise_dataset_names]
noise_manifest = pd.concat(noise_tables, ignore_index=True) if noise_tables else pd.DataFrame()
if noise_manifest.empty:
    raise RuntimeError('No noise files were found in the selected automatic noise sources.')
print('Speech files:', len(speech_manifest), '| Noise files:', len(noise_manifest))

### Save and inspect source manifests

CSV files preserve the auditable source table. The summary confirms which datasets and metadata are active before splitting.

In [ ]:
speech_manifest.to_csv(METADATA_DIR / f'speech_manifest_{CONFIG_ID}.csv', index=False)
noise_manifest.to_csv(METADATA_DIR / f'noise_manifest_{CONFIG_ID}.csv', index=False)
display(speech_manifest.head())
display(noise_manifest.groupby(['dataset', 'noise_category']).size().reset_index(name='count'))
print('PASS: Source manifests contain dataset, source, category, and environment provenance.')

# Stage 4 - Leakage-safe splits and segmentation records

## Step 8 - Split speakers/noise and create segments

Speech speakers and complete noise recordings are assigned to exactly one split before two-second windows are created. This prevents neighboring segments from crossing train, validation, and test boundaries.

In [ ]:
def split_ids(ids, seed, train_fraction=0.70, validation_fraction=0.15):
    values = sorted(map(str, set(ids)))
    rng = np.random.default_rng(seed)
    rng.shuffle(values)
    if len(values) < 3:
        raise ValueError('At least three independent IDs are required for splitting.')
    train_end = max(1, int(len(values) * train_fraction))
    validation_end = min(len(values) - 1, train_end + max(1, int(len(values) * validation_fraction)))
    return {'train': set(values[:train_end]), 'validation': set(values[train_end:validation_end]), 'test': set(values[validation_end:])}

### Execute source splits

Speakers and complete noise recordings are assigned to one split each. A `PASS` requires disjoint source IDs.

In [ ]:
if speech_manifest.speaker_id.nunique() < 3:
    speech_manifest = build_speech_manifest(speech_source['local_path'], limit=10**9, dataset_name=speech_dataset_name)
if speech_manifest.speaker_id.nunique() < 3:
    raise ValueError(f'Fewer than three speakers were found in {speech_dataset_name}.')
speech_split_ids = split_ids(speech_manifest.speaker_id, SEED)
noise_split_ids = split_ids(noise_manifest.noise_source_id, SEED + 1)
speech_manifest['split'] = speech_manifest.speaker_id.map(lambda value: next(name for name, values in speech_split_ids.items() if value in values))
noise_manifest['split'] = noise_manifest.noise_source_id.map(lambda value: next(name for name, values in noise_split_ids.items() if value in values))
print('Speech speakers:', speech_manifest.speaker_id.nunique(), '| Noise sources:', noise_manifest.noise_source_id.nunique())

### Segment records

Two-second windows are generated from already split source files. The records are lazy provenance, so waveform mixtures are not written to disk.

In [ ]:
def make_segments(manifest, source_id_column, segments_per_file, kind):
    rows = []
    for _, item in manifest.iterrows():
        max_start = max(0.0, item.duration - SEGMENT_SECONDS)
        starts = np.linspace(0, max_start, num=max(1, segments_per_file))
        for segment_index, start in enumerate(starts):
            row = item.to_dict()
            row.update({'source_id': str(item[source_id_column]), 'start_time': float(start), 'segment_duration': SEGMENT_SECONDS, 'segment_index': segment_index, 'kind': kind})
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
speech_segments = make_segments(speech_manifest, 'speaker_id', SEGMENTS_PER_SPEECH_FILE, 'speech')
noise_segments = make_segments(noise_manifest, 'noise_source_id', SEGMENTS_PER_NOISE_FILE, 'noise')
print('Speech segments:', len(speech_segments), '| Noise segments:', len(noise_segments))
print('PASS: Segment records were generated after independent source splitting.')

## Step 9 - Standardize audio

The next cell defines resampling, mono loading, padding/truncation, RMS measurement, SNR mixing, and optional RIR processing for every audio example.

In [ ]:
def load_standard_audio(path, start_time=0.0, duration=SEGMENT_SECONDS):
    audio, source_rate = librosa.load(str(path), sr=None, mono=True, offset=float(start_time), duration=float(duration))
    audio = np.asarray(audio, dtype=np.float32)
    if source_rate != SAMPLE_RATE and len(audio):
        audio = librosa.resample(audio, orig_sr=source_rate, target_sr=SAMPLE_RATE).astype(np.float32)
    if len(audio) < SAMPLES_PER_SEGMENT:
        audio = np.pad(audio, (0, SAMPLES_PER_SEGMENT - len(audio)))
    return audio[:SAMPLES_PER_SEGMENT].astype(np.float32)

def rms(audio):
    audio = np.asarray(audio, dtype=np.float64)
    return float(np.sqrt(np.mean(np.square(audio))))

### SNR mixer

For speech $s$ and noise $n$, the target scaling is $\u03b1 = RMS(s)/(RMS(n)10^{SNR_{dB}/20})$. The measured value checks that the requested condition was achieved.

In [ ]:
def mix_at_snr(speech, noise, target_snr_db):
    speech = np.asarray(speech, dtype=np.float32)
    noise = np.asarray(noise, dtype=np.float32)
    speech_level, noise_level = rms(speech), rms(noise)
    if speech_level < 1e-8 or noise_level < 1e-8:
        raise ValueError('Speech and noise must be non-silent for stable SNR.')
    scaling_factor = speech_level / (noise_level * (10 ** (target_snr_db / 20.0)))
    scaled_noise = noise * scaling_factor
    mixture = speech + scaled_noise
    peak = max(float(np.max(np.abs(mixture))), 1e-8)
    if peak > 0.99:
        factor = 0.99 / peak
        speech, scaled_noise, mixture = speech * factor, scaled_noise * factor, mixture * factor
    measured = 20.0 * np.log10(rms(speech) / rms(scaled_noise))
    return mixture.astype(np.float32), float(measured)

### RIR processing and synthetic tests

RIR convolution models room response. The test uses a synthetic decaying impulse response, so it checks numerical length and finiteness without claiming external acoustic validity.

In [ ]:
def normalize_rir(rir):
    rir = np.asarray(rir, dtype=np.float32)
    rir = rir - np.mean(rir)
    return rir / np.sqrt(np.sum(rir ** 2) + 1e-12)

def apply_rir(audio, rir):
    filtered = fftconvolve(audio, normalize_rir(rir), mode='full')[:SAMPLES_PER_SEGMENT]
    peak = max(float(np.max(np.abs(filtered))), 1e-8)
    return (filtered / peak * 0.95).astype(np.float32) if peak > 1 else filtered.astype(np.float32)

def discover_verified_rirs():
    return sorted(path for path in VERIFIED_RIR_DIR.rglob('*.wav') if path.is_file())

In [ ]:
verified_rirs = discover_verified_rirs()
test_speech = np.ones(SAMPLES_PER_SEGMENT, dtype=np.float32) * 0.1
test_noise = np.sin(np.linspace(0, 400 * np.pi, SAMPLES_PER_SEGMENT)).astype(np.float32) * 0.1
snr_checks = [{'target_db': target, 'measured_db': mix_at_snr(test_speech, test_noise, target)[1], 'pass': abs(mix_at_snr(test_speech, test_noise, target)[1] - target) < 0.05} for target in SNR_LEVELS_DB]
rir_unit = apply_rir(test_speech, np.exp(-np.arange(256, dtype=np.float32) / 30.0))
assert np.isfinite(rir_unit).all() and len(rir_unit) == SAMPLES_PER_SEGMENT
display(pd.DataFrame(snr_checks))
print('PASS: SNR and RIR synthetic tests succeeded.')

## Step 10 - Test SNR and RIR mathematics

These small tests use synthetic signals so they do not depend on the downloaded datasets. The requested SNR values should be reproduced within the chosen tolerance, and RIR convolution should return finite audio with the expected length.

A passing SNR test means the noise scaling formula is behaving as intended. A passing RIR test means the convolution code is numerically usable; it does not claim that an external RIR is valid until that RIR has been inspected.

In [ ]:
verified_rirs = discover_verified_rirs()
test_speech = np.ones(SAMPLES_PER_SEGMENT, dtype=np.float32) * 0.1
test_noise = np.sin(np.linspace(0, 400 * np.pi, SAMPLES_PER_SEGMENT)).astype(np.float32) * 0.1
snr_checks = []
for target in SNR_LEVELS_DB:
    _, measured = mix_at_snr(test_speech, test_noise, target)
    snr_checks.append({
        'target_db': target,
        'measured_db': measured,
        'pass': abs(measured - target) < 0.05
    })
rir_unit = apply_rir(
    test_speech,
    np.exp(-np.arange(256, dtype=np.float32) / 30.0)
)
assert np.isfinite(rir_unit).all()
assert len(rir_unit) == SAMPLES_PER_SEGMENT

display(pd.DataFrame(snr_checks))
print(
    'PASS: SNR generation is within tolerance.'
    if all(item['pass'] for item in snr_checks)
    else 'FAIL: SNR generation check failed.'
)
print('PASS: RIR convolution unit test succeeded.')
print('Verified external RIR files:', len(verified_rirs))

# Stage 5 - Build a balanced classification manifest

## Step 11 - Build the classification manifest

**Purpose:** create a balanced, reproducible experiment table without materializing every possible audio mixture on disk.

For each split, the notebook samples matching numbers of clean speech windows, environmental-noise windows, and noisy-speech windows at the configured SNR levels. Each noisy row keeps its source and mixing provenance for later checks and error analysis.

In [ ]:
def rows_for_split(table, split):
    return table[table.split == split].reset_index(drop=True)

def classification_record(split, class_id, speech_item=None, noise_item=None, target_snr_db=np.nan, measured_snr_db=np.nan, rir_filepath='', reverberant=False):
    return {'split': split, 'class_id': class_id, 'class_name': CLASS_NAMES[class_id], 'speech_filepath': str(speech_item.filepath) if speech_item is not None else '', 'speech_source_id': str(speech_item.source_id) if speech_item is not None else '', 'speech_start_time': float(speech_item.start_time) if speech_item is not None else 0.0, 'noise_filepath': str(noise_item.filepath) if noise_item is not None else '', 'noise_source_id': str(noise_item.source_id) if noise_item is not None else '', 'noise_start_time': float(noise_item.start_time) if noise_item is not None else 0.0, 'noise_dataset': str(noise_item.dataset) if noise_item is not None else '', 'noise_category': str(noise_item.noise_category) if noise_item is not None else '', 'target_snr_db': target_snr_db, 'measured_snr_db': measured_snr_db, 'rir_filepath': rir_filepath, 'reverberant': reverberant}

def class_zero_record(split, speech_item):
    return classification_record(split, 0, speech_item=speech_item)

def class_one_record(split, noise_item):
    return classification_record(split, 1, noise_item=noise_item)

def class_two_record(split, speech_item, noise_item, target, rng):
    use_rir = bool(verified_rirs) and rng.random() < RIR_PROBABILITY
    rir_path = str(verified_rirs[int(rng.integers(len(verified_rirs)))]) if use_rir else ''
    speech_audio = load_standard_audio(speech_item.filepath, speech_item.start_time)
    noise_audio = load_standard_audio(noise_item.filepath, noise_item.start_time)
    if use_rir:
        speech_audio = apply_rir(speech_audio, sf.read(rir_path, always_2d=False)[0])
    _, measured = mix_at_snr(speech_audio, noise_audio, target)
    return classification_record(split, 2, speech_item, noise_item, target, measured, rir_path, use_rir)

### Build class 0, class 1, and class 2 records

Classes are balanced within each split. Class 2 stores source paths, target/measured SNR, and optional RIR provenance; the audio mixture remains lazy.

## Generate the balanced classification records

The following cell pairs source records within each split and creates the three labels while retaining paths, start times, SNR values, and reverberation provenance.

In [ ]:
classification_manifest = build_classification_records()

source_paths = classification_manifest[['speech_filepath', 'noise_filepath']].replace('', np.nan).stack()
if not source_paths.map(Path).map(Path.exists).all():
    raise FileNotFoundError('A source file referenced by the classification manifest is missing.')

noisy_rows = classification_manifest[classification_manifest.class_id == 2]
snr_error = (noisy_rows.measured_snr_db - noisy_rows.target_snr_db).abs()
if not snr_error.empty and not (snr_error < 0.05).all():
    raise AssertionError(
        f'Generated noisy-speech SNR exceeded tolerance: max error={snr_error.max():.4f} dB'
    )

classification_manifest.to_csv(
    MANIFEST_DIR / f'classification_manifest_{CONFIG_ID}.csv',
    index=False
)
display(classification_manifest.groupby(['split', 'class_name']).size().unstack(fill_value=0))
display(
    classification_manifest[classification_manifest.class_id == 2]
    .groupby(['split', 'target_snr_db', 'reverberant'])
    .size()
    .reset_index(name='count')
)
print('PASS: Balanced classification manifest saved for config', CONFIG_ID)

# Stage 6 - Explicit leakage and data integrity checks

## Step 12 - Run leakage checks

**Purpose:** turn the most important FYP risk, data leakage, into executable assertions.

The checks verify disjoint speakers and noise sources, existing paths, unique windows, complete class coverage, and requested SNR accuracy. A failure raises an error so the notebook cannot quietly continue with an invalid experiment.

In [ ]:
def check_disjoint_sets(split_map, label):
    split_names = list(split_map)
    for left_index, left_name in enumerate(split_names):
        for right_name in split_names[left_index + 1:]:
            overlap = split_map[left_name] & split_map[right_name]
            if overlap:
                raise AssertionError(f'FAIL: {label} leakage between {left_name} and {right_name}: {list(overlap)[:3]}')
    print(f'PASS: No {label} leakage detected.')

check_disjoint_sets(speech_split_ids, 'speaker')
check_disjoint_sets(noise_split_ids, 'noise-source')
for source_column in ['speech_source_id', 'noise_source_id']:
    source_rows = classification_manifest.loc[classification_manifest[source_column] != '', ['split', source_column]]
    source_split_counts = source_rows.groupby(source_column).split.nunique()
    assert source_split_counts.max() <= 1, f'FAIL: {source_column} appears in multiple dataset splits.'

window_key = ['split', 'speech_filepath', 'noise_filepath', 'speech_source_id', 'noise_source_id', 'speech_start_time', 'noise_start_time', 'class_id', 'target_snr_db']
duplicate_keys = classification_manifest.duplicated(subset=window_key, keep=False)
assert not duplicate_keys.any(), 'FAIL: Duplicate source windows detected.'
print('PASS: Source IDs and generated windows are isolated and non-duplicated.')
assert classification_manifest.groupby('split').class_id.nunique().eq(3).all()
print('PASS: Every split contains all three classes.')

# Stage 7 - Mel features and training-only normalization

## Step 13 - Explain and define mel features

**Purpose:** represent each two-second waveform as a stable time-frequency image that a small CNN can classify.

The mel configuration is defined once near the beginning of the notebook. The feature function calculates the frame count from `n_fft`, `hop_length`, and the `center` policy rather than assuming a fixed shape.

In [ ]:
def expected_feature_shape():
    if FEATURE_CONFIG['center']:
        frames = 1 + SAMPLES_PER_SEGMENT // FEATURE_CONFIG['hop_length']
    else:
        frames = 1 + (SAMPLES_PER_SEGMENT - FEATURE_CONFIG['n_fft']) // FEATURE_CONFIG['hop_length']
    return (FEATURE_CONFIG['n_mels'], frames)

EXPECTED_FEATURE_SHAPE = expected_feature_shape()

def audio_from_record(record):
    if int(record.class_id) == 0:
        return load_standard_audio(record.speech_filepath, record.speech_start_time)
    if int(record.class_id) == 1:
        return load_standard_audio(record.noise_filepath, record.noise_start_time)
    speech = load_standard_audio(record.speech_filepath, record.speech_start_time)
    noise = load_standard_audio(record.noise_filepath, record.noise_start_time)
    if bool(record.reverberant) and record.rir_filepath:
        speech = apply_rir(
            speech,
            sf.read(record.rir_filepath, always_2d=False)[0]
        )
    mixture, _ = mix_at_snr(speech, noise, float(record.target_snr_db))
    return mixture

def extract_mel_feature(audio):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=FEATURE_CONFIG['n_fft'],
        hop_length=FEATURE_CONFIG['hop_length'],
        win_length=FEATURE_CONFIG['win_length'],
        n_mels=FEATURE_CONFIG['n_mels'],
        fmin=FEATURE_CONFIG['fmin'],
        fmax=FEATURE_CONFIG['fmax'],
        power=FEATURE_CONFIG['power'],
        center=FEATURE_CONFIG['center']
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)

def feature_iterator(table):
    for _, record in table.iterrows():
        yield extract_mel_feature(audio_from_record(record)), int(record.class_id), record

## Step 14 - Check feature shape

A mel-spectrogram is a compact picture of how energy changes across frequency and time. The next cell extracts one feature and checks its calculated shape and finite values.

In [ ]:
sample_feature, _, _ = next(
    feature_iterator(classification_manifest.head(1))
)
print('Expected feature shape:', EXPECTED_FEATURE_SHAPE)
print('Actual feature shape:', sample_feature.shape)
assert sample_feature.shape == EXPECTED_FEATURE_SHAPE
assert np.isfinite(sample_feature).all()
print('PASS: Feature shape and finite-value checks succeeded.')

## Step 15 - Calculate training-only normalization

Neural networks usually train more steadily when input features are on a similar numerical scale. The mean and standard deviation are calculated from `train_table` only, then reused unchanged for validation, test, and future inference audio.

In [ ]:
train_table = classification_manifest[
    classification_manifest.split == 'train'
].reset_index(drop=True)
feature_sum = np.zeros(EXPECTED_FEATURE_SHAPE, dtype=np.float64)
feature_square_sum = np.zeros(EXPECTED_FEATURE_SHAPE, dtype=np.float64)
feature_count = 0

for feature, _, _ in tqdm(
    feature_iterator(train_table),
    total=len(train_table),
    desc='Training normalization'
):
    feature_sum += feature
    feature_square_sum += feature.astype(np.float64) ** 2
    feature_count += 1

normalization_mean = feature_sum / max(feature_count, 1)
normalization_std = np.sqrt(
    np.maximum(
        feature_square_sum / max(feature_count, 1) - normalization_mean ** 2,
        1e-6
    )
).astype(np.float32)
normalization = {
    'mean': normalization_mean.tolist(),
    'std': normalization_std.tolist(),
    'source': 'training split only',
    'config_id': CONFIG_ID
}
(CONFIG_DIR / f'normalization_{CONFIG_ID}.json').write_text(
    json.dumps(normalization),
    encoding='utf-8'
)
print('Training records used:', feature_count)
print('PASS: Normalization statistics calculated from training data only.')

# Stage 8 - Lazy TensorFlow datasets and CNN

## Step 16 - Define lazy TensorFlow datasets

**Purpose:** feed one standardized feature at a time to TensorFlow so Colab memory is not consumed by the entire raw dataset.

The next cells define lazy generators and batching. The CNN is introduced and built in the following model section.

In [ ]:
def normalized_feature(record):
    feature = extract_mel_feature(audio_from_record(record))
    return (
        (feature - normalization_mean) / normalization_std
    ).astype(np.float32)[..., np.newaxis]

def tensorflow_generator(table):
    for _, record in table.iterrows():
        yield normalized_feature(record), np.int32(record.class_id)

def make_tf_dataset(table, shuffle=False):
    dataset = tf.data.Dataset.from_generator(
        lambda: tensorflow_generator(table),
        output_signature=(
            tf.TensorSpec(
                shape=(*EXPECTED_FEATURE_SHAPE, 1),
                dtype=tf.float32
            ),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        dataset = dataset.shuffle(
            min(len(table), 1000),
            seed=SEED,
            reshuffle_each_iteration=False
        )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## Create the lazy TensorFlow dataset instances

The generator loads one record, standardizes it, extracts its mel feature, and yields it to TensorFlow. Validation and test records keep fixed order so predictions can be matched back to the manifest.

In [ ]:
validation_table = classification_manifest[
    classification_manifest.split == 'validation'
].reset_index(drop=True)
test_table = classification_manifest[
    classification_manifest.split == 'test'
].reset_index(drop=True)
train_dataset = make_tf_dataset(train_table, shuffle=True)
validation_dataset = make_tf_dataset(validation_table)
test_dataset = make_tf_dataset(test_table)
print('Training records:', len(train_table))
print('Validation records:', len(validation_table))
print('Testing records:', len(test_table))
print('PASS: Lazy TensorFlow datasets are ready.')

## Step 17 - Understand the CNN

The model follows this path:

```text
Audio waveform
    Ã¢â€ â€œ
Mel-spectrogram
    Ã¢â€ â€œ
CNN filters learn time-frequency patterns
    Ã¢â€ â€œ
Dense decision layer
    Ã¢â€ â€œ
Three class probabilities
```

A convolution layer learns small local patterns. ReLU keeps useful positive responses, pooling reduces the feature-map size, batch normalization helps keep activations stable, and dropout randomly removes some connections during training to reduce overfitting. The final softmax layer produces one probability for each class.

## Step 18 - Build the CNN

The next cell creates the compact convolutional classifier, compiles it, and verifies its input and output dimensions.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*EXPECTED_FEATURE_SHAPE, 1)),
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()
assert model.input_shape[1:] == (*EXPECTED_FEATURE_SHAPE, 1)
assert model.output_shape[-1] == 3
print('PASS: Model input and output shapes are correct.')

# Stage 9 - Train and save the best validation model

## Step 19 - Configure training callbacks

The callbacks reduce overfitting, adjust the learning rate, and keep the best validation checkpoint.

In [ ]:
best_model_path = CHECKPOINT_DIR / f'best_classifier_{CONFIG_ID}.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(best_model_path),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5 if not QUICK_TEST_MODE else 1,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]
print('Best checkpoint:', best_model_path)
print('PASS: Training callbacks are configured.')

## Step 20 - Train the model

An epoch is one pass through the training records. The validation set guides model selection but is never used to calculate the test score.

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)
if best_model_path.exists():
    model = tf.keras.models.load_model(best_model_path)

history_json = {
    key: [float(value) for value in values]
    for key, values in history.history.items()
}
(RESULT_DIR / f'training_history_{CONFIG_ID}.json').write_text(
    json.dumps(history_json, indent=2),
    encoding='utf-8'
)
print('PASS: Training finished and best weights were restored when available.')

## Step 21 - Plot training curves

Training and validation accuracy and loss curves show whether the model is learning patterns that generalize beyond its training records. The plots are saved with the experiment configuration ID.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / f'training_curves_{CONFIG_ID}.png', dpi=150)
plt.show()
print('PASS: Training curves were saved.')

# Stage 10 - Untouched test evaluation and robustness analysis

## Step 22 - Evaluate untouched test data

The test report measures generalization on speakers and noise recordings kept outside training. It includes metrics, a confusion matrix, saved predictions, and robustness grouped by noise condition and SNR. Test results do not choose model weights or normalization statistics.

In [ ]:
def predictions_for_table(table):
    probabilities = model.predict(make_tf_dataset(table), verbose=0)
    return probabilities, np.argmax(probabilities, axis=1)

test_probabilities, test_predictions = predictions_for_table(test_table)
test_actual = test_table.class_id.to_numpy()
test_accuracy = float(accuracy_score(test_actual, test_predictions))
report = classification_report(test_actual, test_predictions, labels=[0, 1, 2], target_names=[CLASS_NAMES[i] for i in range(3)], output_dict=True, zero_division=0)
report_table = pd.DataFrame(report).T
display(report_table)
cm = confusion_matrix(test_actual, test_predictions, labels=[0, 1, 2])
plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt='d', cmap='crest', xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values()); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout(); plt.savefig(PLOT_DIR / f'confusion_matrix_{CONFIG_ID}.png', dpi=150); plt.show()
predictions_table = test_table.copy()
predictions_table['predicted_class_id'] = test_predictions
predictions_table['predicted_class'] = [CLASS_NAMES[int(value)] for value in test_predictions]
predictions_table['confidence'] = test_probabilities.max(axis=1)
predictions_table.to_csv(RESULT_DIR / f'test_predictions_{CONFIG_ID}.csv', index=False)

def grouped_accuracy(table, group_columns):
    rows = []
    for group_values, group in table.groupby(group_columns, dropna=False):
        if not isinstance(group_values, tuple): group_values = (group_values,)
        row = dict(zip(group_columns, group_values))
        row.update({'count': len(group), 'accuracy': accuracy_score(group.class_id, group.predicted_class_id)})
        rows.append(row)
    return pd.DataFrame(rows)

noise_dataset_results = grouped_accuracy(predictions_table[predictions_table.class_id == 2], ['noise_dataset'])
noise_category_results = grouped_accuracy(predictions_table[predictions_table.class_id == 2], ['noise_category'])
reverberation_results = grouped_accuracy(predictions_table[predictions_table.class_id == 2], ['reverberant'])
display(noise_dataset_results)
display(noise_category_results)
display(reverberation_results)
noise_dataset_results.to_csv(RESULT_DIR / f'test_by_noise_dataset_{CONFIG_ID}.csv', index=False)
noise_category_results.to_csv(RESULT_DIR / f'test_by_noise_category_{CONFIG_ID}.csv', index=False)
reverberation_results.to_csv(RESULT_DIR / f'test_by_reverberation_{CONFIG_ID}.csv', index=False)

robustness_rows = []
for snr in SNR_LEVELS_DB:
    subset = test_table[(test_table.class_id == 2) & (test_table.target_snr_db == snr)].reset_index(drop=True)
    if subset.empty: continue
    probabilities, predictions = predictions_for_table(subset)
    actual = subset.class_id.to_numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(actual, predictions, labels=[2], average='macro', zero_division=0)
    robustness_rows.append({'snr_db': snr, 'accuracy': accuracy_score(actual, predictions), 'precision': precision, 'recall': recall, 'f1': f1, 'count': len(subset)})
robustness_table = pd.DataFrame(robustness_rows)
display(robustness_table)
if not robustness_table.empty:
    plt.figure(figsize=(7, 4)); plt.plot(robustness_table.snr_db, robustness_table.accuracy, marker='o'); plt.gca().invert_xaxis(); plt.xlabel('Target SNR (dB)'); plt.ylabel('Noisy-speech accuracy'); plt.title('Noisy-Speech Accuracy vs SNR'); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(PLOT_DIR / f'snr_accuracy_{CONFIG_ID}.png', dpi=150); plt.show()
robustness_table.to_csv(RESULT_DIR / f'snr_robustness_{CONFIG_ID}.csv', index=False)
print('Test accuracy:', test_accuracy)
print('PASS: Test evaluation and SNR robustness results were generated.')

## Step 23 - Interpret evaluation metrics

Precision, recall, F1-score, the confusion matrix, and SNR robustness results provide complementary views of performance. The notebook reports measured values without inferring an error cause before inspecting the examples.

# Stage 11 - Listening and prediction demonstrations

## Step 24 - Listen to examples and perform error analysis

The next code cell provides labeled audio players and summarizes unseen test examples, incorrect predictions, SNR, and reverberation status.

In [ ]:
from IPython.display import Audio, display

def normalize_for_playback(audio):
    peak = max(float(np.max(np.abs(audio))), 1e-8)
    return (audio / peak * 0.95).astype(np.float32)

demo_requests = [
    ('Clean speech', classification_manifest[classification_manifest.class_id == 0]),
    ('Environmental noise', classification_manifest[classification_manifest.class_id == 1]),
    ('Non-reverberant noisy speech', classification_manifest[(classification_manifest.class_id == 2) & (~classification_manifest.reverberant)]),
    ('Reverberant + noisy speech', classification_manifest[(classification_manifest.class_id == 2) & (classification_manifest.reverberant)]),
]
for label, candidates in demo_requests:
    if candidates.empty:
        print(f'Demo unavailable in this run: {label}')
        continue
    record = candidates.iloc[0]
    audio = normalize_for_playback(audio_from_record(record))
    print(f'Demo: {label}')
    display(Audio(audio, rate=SAMPLE_RATE))

if verified_rirs:
    reverberant_speech = apply_rir(
        audio_from_record(classification_manifest[classification_manifest.class_id == 0].iloc[0]),
        sf.read(verified_rirs[0], always_2d=False)[0]
    )
    print('Demo: Reverberant speech')
    display(Audio(normalize_for_playback(reverberant_speech), rate=SAMPLE_RATE))
else:
    print('Demo unavailable in this run: Reverberant speech (no verified RIR was supplied).')

display(predictions_table[['class_name', 'predicted_class', 'confidence', 'target_snr_db', 'reverberant']].head(10))
incorrect = predictions_table[predictions_table.class_id != predictions_table.predicted_class_id].copy()
print('Incorrect test examples:', len(incorrect))
if len(incorrect):
    display(incorrect.groupby(['class_name', 'predicted_class']).size().reset_index(name='count').sort_values('count', ascending=False))
    display(incorrect.groupby(['target_snr_db', 'reverberant']).size().reset_index(name='count'))
else:
    print('No incorrect examples in this runtime test set; no failure cause is inferred.')

# Stage 12 - Export for MATLAB R2022 integration

## Step 25 - Export MATLAB and model artifacts

The next cell saves the authoritative Keras model, class mapping, normalization information, and versioned MATLAB preprocessing contract. ONNX export remains optional.

In [ ]:
final_model_path = MODEL_DIR / f'noise_classifier_{CONFIG_ID}.keras'
model.save(final_model_path)
matlab_config = {
    'project_title': PROJECT_TITLE, 'config_id': CONFIG_ID, 'class_ids': CLASS_NAMES,
    'sample_rate': SAMPLE_RATE, 'channels': CHANNELS, 'audio_format': 'mono float32',
    'segment_duration_seconds': SEGMENT_SECONDS, 'samples_per_segment': SAMPLES_PER_SEGMENT,
    'padding_truncation': 'zero-pad at end; truncate at SAMPLES_PER_SEGMENT',
    'n_fft': FEATURE_CONFIG['n_fft'], 'hop_length': FEATURE_CONFIG['hop_length'], 'win_length': FEATURE_CONFIG['win_length'],
    'n_mels': FEATURE_CONFIG['n_mels'], 'fmin': FEATURE_CONFIG['fmin'], 'fmax': FEATURE_CONFIG['fmax'],
    'power': FEATURE_CONFIG['power'], 'center': FEATURE_CONFIG['center'], 'db_conversion': FEATURE_CONFIG['db_conversion'],
    'expected_mel_dimensions': list(EXPECTED_FEATURE_SHAPE), 'expected_cnn_input_dimensions': list((*EXPECTED_FEATURE_SHAPE, 1)),
    'model_input_dtype': 'float32', 'normalization_mean': normalization_mean.tolist(), 'normalization_std': normalization_std.tolist(),
    'model_file': str(final_model_path), 'normalization_source': 'training split only'
}
matlab_config_path = MATLAB_DIR / f'matlab_integration_config_{CONFIG_ID}.json'
matlab_config_path.write_text(json.dumps(matlab_config, indent=2), encoding='utf-8')
(EXPORT_DIR / f'class_mapping_{CONFIG_ID}.json').write_text(json.dumps(CLASS_NAMES, indent=2), encoding='utf-8')
print('PASS: Keras model saved:', final_model_path.exists())
print('PASS: MATLAB configuration saved:', matlab_config_path.exists())

try:
    import tf2onnx
    onnx_path = EXPORT_DIR / f'noise_classifier_{CONFIG_ID}.onnx'
    input_signature = [tf.TensorSpec((None, *EXPECTED_FEATURE_SHAPE, 1), tf.float32, name='mel_input')]
    onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature, opset=13)
    onnx_path.write_bytes(onnx_model.SerializeToString())
    print('OPTIONAL ONNX EXPORT PASS:', onnx_path)
except Exception as error:
    print('OPTIONAL ONNX EXPORT SKIPPED/FAILED:', error)
    print('The Keras model and MATLAB preprocessing contract remain the authoritative artifacts.')

# Stage 13 - Final automatic sanity check and project summary

## Step 26 - Run the final sanity check and summarize the project

The final cell prints PASS or FAIL only for checks executed in this run, reports the experiment configuration and measured metrics, and verifies the exported artifact paths.

In [ ]:
used_speech = [speech_dataset_name] if (classification_manifest.speech_filepath != '').any() else []
used_noise = sorted(set(classification_manifest.loc[classification_manifest.noise_dataset != '', 'noise_dataset']))
used_datasets = sorted(set(used_speech + used_noise))
for dataset_name in used_datasets:
    dataset_mask = ((classification_manifest.speech_filepath != '') & (dataset_name == speech_dataset_name)) | (classification_manifest.noise_dataset == dataset_name)
    for split, column in [('train', 'Used for training?'), ('validation', 'Used for validation?'), ('test', 'Used for testing?')]:
        dataset_inventory.loc[dataset_inventory.Dataset == dataset_name, column] = bool((classification_manifest.loc[dataset_mask, 'split'] == split).any())

status_by_dataset = dict(zip(dataset_inventory.Dataset, dataset_inventory['Acquisition status']))
automatic_statuses = {'AVAILABLE', 'SKIPPED_IN_QUICK_TEST', 'FAILED'}
checks = {
    'Dataset inventory complete': set(DATASET_REGISTRY) == set(dataset_inventory.Dataset) and all(column in dataset_inventory.columns for column in ['Dataset', 'Download capability', 'Acquisition status', 'Number of files', 'Role', 'Used for training?', 'Used for validation?', 'Used for testing?', 'Reason if not used', 'local path', 'source URL', 'license/access']),
    'Automatic status reporting': all(status_by_dataset[name] in automatic_statuses for name, source in DATASET_SOURCES.items() if source['acquisition'] == 'AUTOMATIC'),
    'Dataset acquisition': all(name in ACQUIRED and Path(item['local_path']).exists() for name, item in ACQUIRED.items()),
    'Audio standardization': len(audio_from_record(classification_manifest.iloc[0])) == SAMPLES_PER_SEGMENT,
    'Speaker leakage': all(not (speech_split_ids[a] & speech_split_ids[b]) for a in speech_split_ids for b in speech_split_ids if a < b),
    'Noise leakage': all(not (noise_split_ids[a] & noise_split_ids[b]) for a in noise_split_ids for b in noise_split_ids if a < b),
    'SNR generation': all(item['pass'] for item in snr_checks),
    'RIR processing': np.isfinite(rir_unit).all() and len(rir_unit) == SAMPLES_PER_SEGMENT,
    'Feature shape': sample_feature.shape == EXPECTED_FEATURE_SHAPE,
    'Normalization': normalization['source'] == 'training split only',
    'Model input': model.input_shape[1:] == (*EXPECTED_FEATURE_SHAPE, 1),
    'Model saved': final_model_path.exists(),
    'MATLAB configuration saved': matlab_config_path.exists()
}
for name, passed in checks.items():
    print(f'{name:.<35} ' + ('PASS' if passed else 'FAIL'))
print('\nPROJECT SUMMARY')
print('Project title:', PROJECT_TITLE)
print('Configuration:', CONFIG_ID, '| Quick test mode:', QUICK_TEST_MODE)
print('Acquisition statuses:', ', '.join(f'{name}={status_by_dataset[name]}' for name in DATASET_REGISTRY))
print('Classifier sources actually used:', ', '.join(used_datasets) or 'None')
print('Manual/future resources:', ', '.join(dataset_inventory.loc[dataset_inventory['Download capability'] == 'MANUAL_REQUIRED', 'Dataset']) or 'None')
print('Samples train/validation/test:', *(int((classification_manifest.split == split).sum()) for split in ['train', 'validation', 'test']))
print('Speakers:', speech_manifest.speaker_id.nunique(), '| Classes:', list(CLASS_NAMES.values()))
print('SNR range (dB):', min(SNR_LEVELS_DB), 'to', max(SNR_LEVELS_DB), '| Sample rate:', SAMPLE_RATE, '| Segment seconds:', SEGMENT_SECONDS)
print('Mel shape:', EXPECTED_FEATURE_SHAPE, '| CNN input:', (*EXPECTED_FEATURE_SHAPE, 1))
print('Best validation accuracy:', max(history.history.get('val_accuracy', [float('nan')])), '| Final test accuracy:', test_accuracy)
print('Model location:', final_model_path)
print('MATLAB configuration:', matlab_config_path)
display(dataset_inventory)
if not all(checks.values()):
    raise RuntimeError('One or more required sanity checks failed; inspect the failure above.')